# QMLHEP16 – Quantum Circuit Design with LLMs: Evaluation Task

This notebook demonstrates a closed-loop framework where a Claude LLM agent designs and evaluates quantum circuits through tool calls. Three custom tools are implemented using the Orchestral AI pattern: (1) a **Meyer-Wallach entanglement measure** tool that quantifies circuit expressibility—a metric central to the QMLHEP16 project; (2) a **QNN training tool** that trains a 4-qubit variational circuit on MNIST binary classification, returning loss and accuracy for agent feedback; and (3) a **circuit architecture search** where the agent simultaneously optimizes learning rate and circuit depth. The agent reasons over returned metrics across multiple rounds and converges on a configuration that balances expressibility and trainability.

In [ ]:
import subprocess, sys
pkgs = ['orchestral-ai', 'pennylane', 'torch', 'torchvision',
        'anthropic', 'scikit-learn', 'matplotlib', 'numpy']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs, check=True)
print('all packages ready')

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import pennylane as qml

from sklearn.decomposition import PCA
import anthropic

In [ ]:
TOOLS = {}

def define_tool(fn):
    TOOLS[fn.__name__] = fn
    return fn

def get_tool_schemas():
    import inspect
    schemas = []
    for name, fn in TOOLS.items():
        sig = inspect.signature(fn)
        props, required = {}, []
        for pname, param in sig.parameters.items():
            ann = param.annotation
            ptype = 'integer' if ann == int else 'number' if ann == float else 'string'
            props[pname] = {'type': ptype}
            if param.default is inspect.Parameter.empty:
                required.append(pname)
        schemas.append({
            'name': name,
            'description': fn.__doc__,
            'input_schema': {'type': 'object', 'properties': props, 'required': required}
        })
    return schemas

print('Tool registry ready')

In [ ]:
@define_tool
def meyer_wallach_entanglement(n_qubits: int, n_layers: int) -> str:
    """Compute the Meyer-Wallach global entanglement measure Q for a random
    variational quantum circuit with n_qubits qubits and n_layers layers of
    Ry rotations and CNOT entanglers. Q in [0,1]: 0=separable, 1=maximally
    entangled. Use this to assess circuit expressibility before training.
    """
    dev = qml.device('default.qubit', wires=n_qubits)

    @qml.qnode(dev)
    def circuit(params):
        for l in range(n_layers):
            for q in range(n_qubits):
                qml.RY(params[l, q], wires=q)
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])
        return qml.state()

    Q_vals = []
    for _ in range(50):
        params = np.random.uniform(0, 2*np.pi, (n_layers, n_qubits))
        state = circuit(params)
        total = 0.0
        for k in range(n_qubits):
            keep = [i for i in range(n_qubits) if i != k]
            rho_k = qml.math.partial_trace(np.outer(state, state.conj()), keep, n_qubits)
            total += 1.0 - float(np.real(np.trace(rho_k @ rho_k)))
        Q_vals.append((2.0 / n_qubits) * total)

    Q_mean = float(np.mean(Q_vals))
    Q_std  = float(np.std(Q_vals))
    return json.dumps({
        'n_qubits': n_qubits,
        'n_layers': n_layers,
        'Q_mean': round(Q_mean, 4),
        'Q_std':  round(Q_std, 4),
        'interpretation': 'high' if Q_mean > 0.5 else 'low'
    })

print(json.loads(meyer_wallach_entanglement(4, 2)))

In [ ]:
def load_mnist_binary(n_train=300, n_test=100):
    tf = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.1307,), (0.3081,))])
    train_full = datasets.MNIST('./data', train=True,  download=True, transform=tf)
    test_full  = datasets.MNIST('./data', train=False, download=True, transform=tf)

    def binary_idx(ds, n):
        return Subset(ds, [i for i, (_, y) in enumerate(ds) if y in (0, 1)][:n])

    X_tr, y_tr, X_te, y_te = [], [], [], []
    for x, y in DataLoader(binary_idx(train_full, n_train), batch_size=n_train):
        X_tr = x.numpy().reshape(len(x), -1); y_tr = y.numpy()
    for x, y in DataLoader(binary_idx(test_full, n_test), batch_size=n_test):
        X_te = x.numpy().reshape(len(x), -1); y_te = y.numpy()

    pca = PCA(n_components=4)
    X_tr = pca.fit_transform(X_tr)
    X_te = pca.transform(X_te)
    for arr in [X_tr, X_te]:
        arr[:] = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * np.pi

    return X_tr, y_tr, X_te, y_te

X_train, y_train, X_test, y_test = load_mnist_binary()
print(f'train={X_train.shape}, test={X_test.shape}')

In [ ]:
N_QUBITS = 4
dev_qnn  = qml.device('default.qubit', wires=N_QUBITS)
_training_history = []

def make_qnode(n_layers):
    @qml.qnode(dev_qnn, interface='torch')
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
        for l in range(n_layers):
            for q in range(N_QUBITS):
                qml.RY(weights[l, q, 0], wires=q)
                qml.RZ(weights[l, q, 1], wires=q)
            for q in range(N_QUBITS - 1):
                qml.CNOT(wires=[q, q + 1])
        return qml.expval(qml.PauliZ(0))
    return circuit

@define_tool
def train_qnn(epochs: int, learning_rate: float, n_layers: int) -> str:
    """Train a variational QNN on MNIST binary classification (digits 0 vs 1).
    The circuit uses AngleEmbedding on 4 qubits then n_layers of RY/RZ rotations
    with CNOT entanglers. Returns train_loss, train_acc, test_acc. Use these
    metrics to decide next learning_rate and n_layers values.
    Good ranges: learning_rate 0.001-0.1, n_layers 1-4, epochs 3-10.
    """
    circuit = make_qnode(n_layers)
    weights = torch.nn.Parameter(
        torch.tensor(np.random.uniform(0, 2*np.pi, (n_layers, N_QUBITS, 2)),
                     dtype=torch.float32)
    )
    optimizer = torch.optim.Adam([weights], lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    X_te = torch.tensor(X_test,  dtype=torch.float32)
    y_te = torch.tensor(y_test,  dtype=torch.float32)

    for _ in range(epochs):
        optimizer.zero_grad()
        preds = torch.stack([circuit(x, weights) for x in X_tr])
        loss  = criterion(preds, 2*y_tr - 1)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        tr_p = torch.stack([circuit(x, weights) for x in X_tr])
        te_p = torch.stack([circuit(x, weights) for x in X_te])
        result = {
            'epochs': epochs, 'learning_rate': learning_rate, 'n_layers': n_layers,
            'train_loss': round(criterion(tr_p, 2*y_tr-1).item(), 4),
            'train_acc':  round(((tr_p > 0).float() == y_tr).float().mean().item(), 4),
            'test_acc':   round(((te_p > 0).float() == y_te).float().mean().item(), 4),
        }
    _training_history.append(result)
    return json.dumps(result)

print(json.loads(train_qnn(3, 0.05, 2)))

In [ ]:
def run_agent(system_prompt, user_message, max_rounds=10):
    client   = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
    schemas  = get_tool_schemas()
    messages = [{'role': 'user', 'content': user_message}]
    log      = []

    for round_i in range(max_rounds):
        response = client.messages.create(
            model='claude-opus-4-6',
            max_tokens=1024,
            system=system_prompt,
            tools=schemas,
            messages=messages,
        )

        if response.stop_reason == 'end_turn':
            final = next((b.text for b in response.content if hasattr(b, 'text')), '')
            log.append({'round': round_i, 'type': 'final', 'text': final})
            break

        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                out = TOOLS[block.name](**block.input)
                parsed = json.loads(out) if out.startswith('{') else out
                log.append({'round': round_i, 'tool': block.name,
                            'input': block.input, 'output': parsed})
                print(f'[round {round_i}] {block.name}({block.input})')
                print(f'  -> {parsed}')
                tool_results.append({'type': 'tool_result',
                                     'tool_use_id': block.id, 'content': out})

        messages.append({'role': 'assistant', 'content': response.content})
        messages.append({'role': 'user',      'content': tool_results})

    return log

print('Agent runner ready')

## Task 1 – Meyer-Wallach Hello World

In [ ]:
log1 = run_agent(
    'You are a quantum computing assistant. Call meyer_wallach_entanglement '
    'to compare circuit expressibility. Be concise.',
    'Compare the entanglement of a 4-qubit circuit with 1 layer versus 3 layers. '
    'Which architecture is more expressive and why?'
)
print()
print(log1[-1]['text'])

## Task 2 – Agent Calls QNN Training Tool

In [ ]:
log2 = run_agent(
    'You are a quantum ML researcher. Use train_qnn to train a QNN on MNIST. '
    'Report the final test accuracy clearly.',
    'Train the QNN with 5 epochs, learning_rate=0.05, n_layers=2 and report results.'
)
print()
print(log2[-1]['text'])

## Task 3 – Agent Searches Learning Rate & Circuit Depth

In [ ]:
_training_history.clear()

log3 = run_agent(
    'You are optimizing a QNN for MNIST binary classification. '
    'Call train_qnn varying BOTH learning_rate (0.001 to 0.1) AND n_layers (1-4). '
    'Use 5 epochs per run. After each result, reason about whether to increase or '
    'decrease each parameter. Run at least 5 experiments then state the best config.',
    'Find the best combination of learning_rate and n_layers. Be strategic.',
    max_rounds=12
)
print()
print(log3[-1]['text'])

In [ ]:
if _training_history:
    lrs      = [r['learning_rate'] for r in _training_history]
    layers   = [r['n_layers']      for r in _training_history]
    test_acc = [r['test_acc']      for r in _training_history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sc = axes[0].scatter(lrs, test_acc, c=layers, cmap='viridis',
                         s=120, edgecolors='k', zorder=3)
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Learning rate (log)')
    axes[0].set_ylabel('Test accuracy')
    axes[0].set_title('Agent search: LR vs Accuracy')
    axes[0].grid(alpha=0.3)
    plt.colorbar(sc, ax=axes[0], label='n_layers')

    for i, r in enumerate(_training_history):
        axes[1].plot(i+1, r['test_acc'], 'o',
                     color=plt.cm.viridis(r['n_layers']/4), ms=10)
    best_acc = max(test_acc)
    axes[1].axhline(best_acc, ls='--', color='red', label=f'best={best_acc:.3f}')
    axes[1].set_xlabel('Experiment #')
    axes[1].set_ylabel('Test accuracy')
    axes[1].set_title('Agent improvement over rounds')
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    best = max(_training_history, key=lambda r: r['test_acc'])
    print(f"Best: lr={best['learning_rate']}, layers={best['n_layers']}, "
          f"test_acc={best['test_acc']:.4f}")

    plt.tight_layout()
    plt.show()